# ARC-AGI-3 LLM Agent v7.0

**New in v7.0:**
- Goal classifier (8 classes) - detects game type in runtime
- Counter-picking: strategy hints per goal class
- Heuristic ValueGate (no ML, no weights)
- Escalation ladder (audit #2+ = STRATEGY RESET)
- Robust LEVEL NOTES extraction

**Expected: ~42-50 score** (threshold: 40.97)

In [ ]:
# Install dependencies
!pip install -q openai numpy pillow arcengine

import os
os.environ['KAGGLE_IS_COMPETITION_RERUN'] = '1'

# LLM configuration - adjust to available model
os.environ.setdefault('LLM_BASE_URL', 'http://127.0.0.1:1234/v1')
os.environ.setdefault('LLM_MODEL_ID', 'qwen2.5-7b')
os.environ.setdefault('LLM_API_KEY', 'EMPTY')

In [ ]:
# Grid utilities
"""Grid formatting and segmentation utilities for ARC-AGI-3 LLM agent.

Derived from the Duck harness (MIT license, Tufa Labs).
Simplified for use with the ARC-AGI-3-Agents framework.
"""
from __future__ import annotations

import hashlib
from typing import Any, Sequence

# ARC color palette: index -> single-char label
ARC_COLOR_CHARS = "WwgGcBMPRbSYOrNp"
ARC_COLOR_LEGEND = (
    "W=white, w=light gray, g=gray, G=dark gray, c=charcoal, B=black, "
    "M=magenta, P=pink, R=red, b=blue, S=sky blue, Y=yellow, O=orange, "
    "r=dark red, N=light green, p=purple"
)

# ARC color palette: index -> RGB
ARC_COLOR_MAP: dict[int, tuple[int, int, int]] = {
    0: (255, 255, 255), 1: (204, 204, 204), 2: (153, 153, 153),
    3: (102, 102, 102), 4: (51, 51, 51),    5: (0, 0, 0),
    6: (229, 58, 163),  7: (255, 123, 204), 8: (249, 60, 49),
    9: (30, 147, 255),  10: (136, 216, 241), 11: (255, 220, 0),
    12: (255, 133, 27), 13: (146, 18, 49),  14: (79, 204, 48),
    15: (163, 86, 214),
}


def format_grid_ascii(grid: Sequence[Sequence[int]]) -> str:
    """Render a numeric grid as ASCII art using ARC color characters."""
    if not grid:
        return "(empty grid)"
    lines = []
    for row in grid:
        chars = []
        for v in row:
            chars.append(ARC_COLOR_CHARS[max(0, min(15, int(v)))])
        lines.append("".join(chars))
    return "\n".join(lines)


def grid_to_png_bytes(grid: Sequence[Sequence[int]], upscale: int = 4) -> bytes:
    """Render a numeric grid as a PNG image (no PIL dependency — pure bytes)."""
    try:
        from PIL import Image
        import io
    except ImportError:
        return b""
    
    rows = len(grid)
    cols = max((len(r) for r in grid), default=0) if rows else 0
    if rows <= 0 or cols <= 0:
        return b""
    
    image = Image.new("RGB", (cols, rows), ARC_COLOR_MAP[0])
    pixels = image.load()
    for row_idx, row in enumerate(grid):
        for col_idx in range(cols):
            value = row[col_idx] if col_idx < len(row) else 0
            pixels[col_idx, row_idx] = ARC_COLOR_MAP.get(int(value), ARC_COLOR_MAP[0])
    
    if upscale > 1:
        image = image.resize((cols * upscale, rows * upscale), Image.Resampling.NEAREST)
    
    buffer = io.BytesIO()
    image.save(buffer, format="PNG")
    return buffer.getvalue()


# ---------------------------------------------------------------------------
# Segmentation (connected-component analysis)
# ---------------------------------------------------------------------------

_ORTH = ((-1, 0), (1, 0), (0, -1), (0, 1))
_CW = ((-1, -1), (-1, 0), (-1, 1), (0, 1), (1, 1), (1, 0), (1, -1), (0, -1))
_CW_INDEX = {off: i for i, off in enumerate(_CW)}


def _trace_outer_contour(cells, start):
    """Moore-neighbour trace of outer perimeter, clockwise."""
    if len(cells) == 1:
        return [start]
    contour = [start]
    b = start
    prev = (start[0], start[1] - 1)
    second = None
    for _ in range(8 * len(cells) + 16):
        idx = _CW_INDEX[(prev[0] - b[0], prev[1] - b[1])]
        nxt = None
        for k in range(1, 9):
            off = _CW[(idx + k) % 8]
            cand = (b[0] + off[0], b[1] + off[1])
            if cand in cells:
                nxt = cand
                back = _CW[(idx + k - 1) % 8]
                new_prev = (b[0] + back[0], b[1] + back[1])
                break
        if nxt is None:
            break
        if second is None:
            second = nxt
        elif b == start and nxt == second:
            break
        contour.append(nxt)
        prev, b = new_prev, nxt
    if len(contour) > 1 and contour[-1] == contour[0]:
        contour.pop()
    return contour


def _corner_points(contour):
    """Reduce contour to direction-change points."""
    if len(contour) <= 2:
        return list(contour)
    m = len(contour)
    corners = []
    for i in range(m):
        prev_c, cur, nxt = contour[i - 1], contour[i], contour[(i + 1) % m]
        d_in = (cur[0] - prev_c[0], cur[1] - prev_c[1])
        d_out = (nxt[0] - cur[0], nxt[1] - cur[1])
        if d_in != d_out:
            corners.append(cur)
    return corners


def _object_hash(cells, color):
    """Translation-invariant signature of an object."""
    min_r = min(r for r, _ in cells)
    min_c = min(c for _, c in cells)
    norm = sorted((r - min_r, c - min_c) for r, c in cells)
    payload = repr((color, norm)).encode()
    return hashlib.sha1(payload).hexdigest()[:16]


def segment_grid(grid: Sequence[Sequence[int]]) -> dict[str, Any]:
    """Segment a grid into connected components.
    
    Returns dict with:
      - nodes: list of {id, color, hash, pixels, boundary, children}
      - adjacency_list: list of [i, j] pairs
    """
    height = len(grid)
    width = len(grid[0]) if height else 0
    if not height or not width:
        return {"nodes": [], "adjacency_list": []}
    
    # Connected components via flood fill
    comp_id = [[-1] * width for _ in range(height)]
    components = []
    
    for sr in range(height):
        for sc in range(width):
            if comp_id[sr][sc] != -1:
                continue
            value = grid[sr][sc]
            cid = len(components)
            cells = set()
            stack = [(sr, sc)]
            while stack:
                r, c = stack.pop()
                if (r, c) in cells:
                    continue
                if 0 <= r < height and 0 <= c < width and grid[r][c] == value and comp_id[r][c] == -1:
                    cells.add((r, c))
                    comp_id[r][c] = cid
                    for dr, dc in _ORTH:
                        nr, nc = r + dr, c + dc
                        if 0 <= nr < height and 0 <= nc < width and comp_id[nr][nc] == -1 and grid[nr][nc] == value:
                            stack.append((nr, nc))
            components.append({"value": value, "cells": cells, "start": (sr, sc)})
    
    # Build nodes
    nodes = []
    for cid, comp in enumerate(components):
        color_char = ARC_COLOR_CHARS[max(0, min(15, comp["value"]))]
        obj_hash = _object_hash(comp["cells"], comp["value"])
        contour = _trace_outer_contour(comp["cells"], comp["start"])
        boundary = _corner_points(contour)
        nodes.append({
            "id": cid,
            "color": color_char,
            "hash": obj_hash,
            "pixels": len(comp["cells"]),
            "boundary": [[r, c] for r, c in boundary[:12]],  # limit size
            "children": [],
        })
    
    # Compute containment (children)
    for outer_id, outer_comp in enumerate(components):
        outer_cells = outer_comp["cells"]
        for inner_id, inner_comp in enumerate(components):
            if inner_id == outer_id:
                continue
            # Check if inner is fully enclosed by outer
            inner_cells = inner_comp["cells"]
            if all(any((r + dr, c + dc) in outer_cells for dr, dc in _ORTH)
                   for r, c in inner_cells):
                # Check no path from inner to grid edge without crossing outer
                # Simple check: all neighbors of inner cells are either inner or outer
                is_child = True
                for r, c in inner_cells:
                    for dr, dc in _ORTH:
                        nr, nc = r + dr, c + dc
                        if 0 <= nr < height and 0 <= nc < width:
                            if comp_id[nr][nc] != outer_id and comp_id[nr][nc] != inner_id:
                                is_child = False
                                break
                    if not is_child:
                        break
                if is_child:
                    nodes[outer_id]["children"].append(inner_id)
    
    # Compute adjacency
    adj_set = set()
    for r in range(height):
        for c in range(width):
            cid = comp_id[r][c]
            for dr, dc in _ORTH:
                nr, nc = r + dr, c + dc
                if 0 <= nr < height and 0 <= nc < width:
                    nid = comp_id[nr][nc]
                    if cid != nid:
                        pair = (min(cid, nid), max(cid, nid))
                        adj_set.add(pair)
    
    return {
        "nodes": nodes,
        "adjacency_list": sorted(adj_set),
    }


def compact_segmentation_summary(seg: dict, max_nodes: int = 20) -> str:
    """Create a compact text summary of segmentation for LLM context."""
    nodes = seg.get("nodes", [])
    adj = seg.get("adjacency_list", [])
    
    lines = [f"Objects: {len(nodes)}, Adjacencies: {len(adj)}"]
    for node in nodes[:max_nodes]:
        boundary_str = ",".join(f"{r},{c}" for r, c in node["boundary"][:4])
        children_str = f", children=[{','.join(map(str, node['children']))}]" if node["children"] else ""
        lines.append(
            f"  obj#{node['id']}: color={node['color']}, size={node['pixels']}px, "
            f"hash={node['hash'][:8]}, corners=[{boundary_str}]{children_str}"
        )
    if len(nodes) > max_nodes:
        lines.append(f"  ... and {len(nodes) - max_nodes} more objects")
    return "\n".join(lines)


In [ ]:
# Goal classifier (8 classes + counter-picking)
"""
ARC-AGI-3 Goal Classifier - detekcja klasy gry w runtime

Inspiracja: c87f7a48-pokemon (Pokemon TCG) - FROSLASS_IDS, LOW_E_AGGRO_IDS
W Pokemon: detekcja archetypu przeciwnika w runtime, zmiana strategii
W ARC-AGI-3: detekcja klasy gry w runtime, zmiana parametrów heurystyki

Klasy gier (z GOAL_CATALOG.md):
1. COVER: pokryj wszystkie znaczniki
2. REACH: dotrzyj do wszystkich pozycji
3. COPY: skopiuj wzorzec
4. REDUCE: zostaw dokładnie 1 element
5. PLACE: umieść wszystkie obiekty w strefie
6. FLOW: wypełnij wszystkie naczynia
7. ALIGN: wyrównaj pozycje według koloru
8. LOCAL_CONSTRAINTS: każdy kafel-klucz 3×3 mówi o sąsiadach

Counter-picking: zmiana parametrów heurystyki na podstawie klasy gry
"""

from dataclasses import dataclass
from typing import Any


# ---------------------------------------------------------------------------
# Goal Classifier - detekcja klasy gry w runtime
# ---------------------------------------------------------------------------

@dataclass
class GameFeatures:
    """Cechy gry używane do detekcji klasy."""
    
    # Pokrycie
    small_markers_count: int = 0         # Małe markery (kontury, kropki)
    outlined_positions_count: int = 0    # Wyznaczone pozycje (outlined)
    colored_edge_markers_count: int = 0  # Kolorowe markery na krawędziach
    
    # Wzorce
    pattern_in_corner: bool = False      # Wzorzec w lewym górnym rogu
    pattern_similarity: float = 0.0      # Podobieństwo do wzorca
    
    # Obiekty
    pieces_count: int = 0                # Pionki (jak w samotniku)
    empty_cells_count: int = 0           # Puste komórki
    boxes_count: int = 0                 # Skrzynki
    target_zone_detected: bool = False   # Strefa celu
    
    # Mechaniki
    dispenser_detected: bool = False     # Dozownik (FLOW)
    vessels_count: int = 0               # Naczynia
    tile_keys_3x3_count: int = 0         # Kafelki-klucze 3×3
    
    # Grid
    grid_height: int = 0
    grid_width: int = 0
    unique_colors: int = 0
    non_zero_cells: int = 0


def extract_game_features(grid: list[list[int]]) -> GameFeatures:
    """Ekstrahuj cechy z gridu do detekcji klasy gry."""
    if not grid or not grid[0]:
        return GameFeatures()
    
    height = len(grid)
    width = len(grid[0])
    
    # Podstawowe cechy gridu
    flat = [cell for row in grid for cell in row]
    unique_colors = len(set(flat))
    non_zero_cells = sum(1 for cell in flat if cell != 0)
    empty_cells = sum(1 for cell in flat if cell == 0)
    
    # Detekcja wzorca w rogu (COPY)
    # Sprawdzamy czy lewy górny róg (10×10) jest wzorcem
    # BUGFIX #1: Sprawdź czy wzorzec NIE jest pusty (0 vs 0 = fałszywe dopasowanie)
    # BUGFIX #2: Sprawdź czy wzorzec ma znaczące non-zero cells (>10%)
    pattern_in_corner = False
    pattern_similarity = 0.0
    if height >= 10 and width >= 10:
        pattern = [row[:10] for row in grid[:10]]
        pattern_flat = [cell for row in pattern for cell in row]
        pattern_non_zero = sum(1 for cell in pattern_flat if cell != 0)
        pattern_density = pattern_non_zero / 100.0
        
        # Tylko jeśli wzorzec jest znaczący (>10% non-zero)
        if pattern_density > 0.1:
            # Porównaj z resztą gridu
            max_sim = 0.0
            for r in range(0, height-9, 5):
                for c in range(0, width-9, 5):
                    if r == 0 and c == 0:
                        continue
                    matches = 0
                    total = 0
                    for pr in range(10):
                        for pc in range(10):
                            if r+pr < height and c+pc < width:
                                # BUGFIX #1: Nie licz 0 vs 0 jako dopasowanie
                                if grid[r+pr][c+pc] == pattern[pr][pc] and pattern[pr][pc] != 0:
                                    matches += 1
                                # Licz tylko non-zero cells we wzorcu
                                if pattern[pr][pc] != 0:
                                    total += 1
                    if total > 0:
                        sim = matches / total
                        max_sim = max(max_sim, sim)
            pattern_similarity = max_sim
            pattern_in_corner = max_sim > 0.7  # 70% podobieństwa = wzorzec
    
    # Detekcja małych markerów (COVER)
    # Małe markery = izolowane komórki (sąsiedzi != kolor)
    small_markers = 0
    for r in range(height):
        for c in range(width):
            if grid[r][c] == 0:
                continue
            neighbors = []
            for dr, dc in [(-1,0), (1,0), (0,-1), (0,1)]:
                nr, nc = r+dr, c+dc
                if 0 <= nr < height and 0 <= nc < width:
                    neighbors.append(grid[nr][nc])
            # Izolowana komórka (wszyscy sąsiedzi != kolor) = mały marker
            if grid[r][c] not in neighbors:
                small_markers += 1
    
    # Detekcja wyznaczonych pozycji (REACH)
    # Outlined positions = komórki otoczone innym kolorem
    outlined_positions = 0
    for r in range(height):
        for c in range(width):
            if grid[r][c] == 0:
                continue
            # Sprawdź czy jest otoczona innym kolorem
            neighbor_colors = set()
            for dr, dc in [(-1,0), (1,0), (0,-1), (0,1)]:
                nr, nc = r+dr, c+dc
                if 0 <= nr < height and 0 <= nc < width:
                    neighbor_colors.add(grid[nr][nc])
            if len(neighbor_colors) > 1 and grid[r][c] not in neighbor_colors:
                outlined_positions += 1
    
    # Detekcja kolorowych markerów na krawędziach (ALIGN)
    colored_edge_markers = 0
    for r in range(height):
        for c in range(width):
            if r == 0 or r == height-1 or c == 0 or c == width-1:
                if grid[r][c] != 0:
                    colored_edge_markers += 1
    
    # Detekcja skrzynek (PLACE)
    # Skrzynki = duże bloki tego samego koloru (5+ komórek)
    visited = [[False] * width for _ in range(height)]
    boxes = 0
    for r in range(height):
        for c in range(width):
            if visited[r][c] or grid[r][c] == 0:
                continue
            # BFS
            queue = [(r, c)]
            component = []
            color = grid[r][c]
            while queue:
                cr, cc = queue.pop(0)
                if visited[cr][cc] or grid[cr][cc] != color:
                    continue
                visited[cr][cc] = True
                component.append((cr, cc))
                for dr, dc in [(-1,0), (1,0), (0,-1), (0,1)]:
                    nr, nc = cr+dr, cc+dc
                    if 0 <= nr < height and 0 <= nc < width:
                        queue.append((nr, nc))
            if len(component) >= 5:  # Skrzynka = 5+ komórek
                boxes += 1
    
    # Detekcja strefy celu (PLACE)
    # Strefa celu = prostokątny obszar z innym kolorem
    target_zone = False
    # Proste heurystyki: jeśli jest duży prostokątny obszar jednego koloru
    # (implementacja uproszczona)
    if boxes > 0 and non_zero_cells > height * width * 0.5:
        target_zone = True
    
    # Detekcja dozownika (FLOW)
    # Dozownik = rząd komórek na górze (wiersz 0) z jednym kolorem
    dispenser = False
    if height > 0 and width > 0:
        top_row = grid[0]
        top_colors = set(c for c in top_row if c != 0)
        if len(top_colors) == 1 and len(top_row) > 5:
            dispenser = True
    
    # Detekcja naczyń (FLOW)
    # Naczynia = małe prostokątne obszary na dole
    vessels = 0
    if height > 5:
        bottom_rows = grid[-5:]
        for r in range(len(bottom_rows)):
            for c in range(width):
                if bottom_rows[r][c] != 0:
                    vessels += 1
        vessels = max(0, vessels // 10)  # Normalizacja
    
    # Detekcja kafelków-kluczy 3×3 (LOCAL_CONSTRAINTS)
    tile_keys = 0
    for r in range(1, height-1):
        for c in range(1, width-1):
            # Sprawdź czy jest 3×3 blok z centralną komórką
            if grid[r][c] != 0:
                is_key = True
                for dr in [-1, 0, 1]:
                    for dc in [-1, 0, 1]:
                        if grid[r+dr][c+dc] == 0:
                            is_key = False
                            break
                    if not is_key:
                        break
                if is_key:
                    tile_keys += 1
    
    # Detekcja pionków (REDUCE)
    # Pionki = małe izolowane obiekty (1-4 komórki)
    visited = [[False] * width for _ in range(height)]
    pieces = 0
    for r in range(height):
        for c in range(width):
            if visited[r][c] or grid[r][c] == 0:
                continue
            # BFS
            queue = [(r, c)]
            component = []
            color = grid[r][c]
            while queue:
                cr, cc = queue.pop(0)
                if visited[cr][cc] or grid[cr][cc] != color:
                    continue
                visited[cr][cc] = True
                component.append((cr, cc))
                for dr, dc in [(-1,0), (1,0), (0,-1), (0,1)]:
                    nr, nc = cr+dr, cc+dc
                    if 0 <= nr < height and 0 <= nc < width:
                        queue.append((nr, nc))
            if 1 <= len(component) <= 4:  # Pionek = 1-4 komórki
                pieces += 1
    
    return GameFeatures(
        small_markers_count=small_markers,
        outlined_positions_count=outlined_positions,
        colored_edge_markers_count=colored_edge_markers,
        pattern_in_corner=pattern_in_corner,
        pattern_similarity=pattern_similarity,
        pieces_count=pieces,
        empty_cells_count=empty_cells,
        boxes_count=boxes,
        target_zone_detected=target_zone,
        dispenser_detected=dispenser,
        vessels_count=vessels,
        tile_keys_3x3_count=tile_keys,
        grid_height=height,
        grid_width=width,
        unique_colors=unique_colors,
        non_zero_cells=non_zero_cells,
    )


def detect_goal_class(grid: list[list[int]]) -> str:
    """Detekcja klasy gry na podstawie wizualnych cech.
    
    Returns:
        COVER, COPY, REACH, REDUCE, PLACE, FLOW, ALIGN, LOCAL_CONSTRAINTS, UNKNOWN
    """
    features = extract_game_features(grid)
    
    # Priorytet: najpierw najbardziej specyficzne
    if features.tile_keys_3x3_count > 3:
        return "LOCAL_CONSTRAINTS"
    
    if features.dispenser_detected and features.vessels_count > 2:
        return "FLOW"
    
    if features.pattern_in_corner:
        return "COPY"
    
    if features.boxes_count > 2 and features.target_zone_detected:
        return "PLACE"
    
    if features.pieces_count > 20 and features.empty_cells_count > features.pieces_count:
        return "REDUCE"
    
    if features.colored_edge_markers_count > 5:
        return "ALIGN"
    
    if features.outlined_positions_count > 5:
        return "REACH"
    
    if features.small_markers_count > 10:
        return "COVER"
    
    return "UNKNOWN"


# ---------------------------------------------------------------------------
# Counter-picking - zmiana parametrów heurystyki na podstawie klasy gry
# ---------------------------------------------------------------------------

@dataclass
class StrategyParams:
    """Parametry strategii dla danej klasy gry."""
    
    explore_weight: float = 1.0
    exploit_weight: float = 2.0
    noop_penalty: float = -1.0
    level_progress_weight: float = 5.0
    affordance_weight: float = 1.5
    pattern_weight: float = 2.0
    stuck_threshold: int = 40
    temp_base: float = 0.6
    batch_size: int = 4


# Counter-picking na poziomie promptu (jak druga gałąź v6.9)
STRATEGY_HINTS = {
    "COVER": "Focus on finding and covering ALL small markers (dots, outlines). Click on each marker once.",
    "COPY": "Find the pattern in the top-left corner. Replicate it exactly in the target area. Don't click the pattern itself.",
    "REACH": "Navigate avatar(s) to all outlined goal positions. Watch for obstacles and traps.",
    "REDUCE": "Jump pieces over each other (like checkers). Goal: leave exactly 1 piece remaining.",
    "PLACE": "Push all boxes into the target zone. Boxes must be RELEASED in the zone (not held).",
    "FLOW": "Set up the configuration FIRST, then CONFIRM (action 5). Limited attempts - don't waste them.",
    "ALIGN": "Align colored markers to match target positions by color. Click markers to move them.",
    "LOCAL_CONSTRAINTS": "Each 3×3 tile-key shows constraints for neighbors. Click to change colors cyclically.",
    "UNKNOWN": "Explore systematically. Test different actions to understand the mechanics. Take notes.",
}


def get_strategy_for_goal_class(goal_class: str) -> StrategyParams:
    """Zwróć parametry strategii dla danej klasy gry.
    
    Inspiracja: c87f7a48-pokemon - FROSLASS counter, LOW_E_AGGRO counter
    """
    strategies = {
        "COVER": StrategyParams(
            explore_weight=1.5,
            exploit_weight=1.8,
            affordance_weight=2.5,  # Klikaj na markery
            stuck_threshold=35,
        ),
        "COPY": StrategyParams(
            explore_weight=0.8,
            exploit_weight=2.2,
            pattern_weight=3.5,  # Priorytet: kopiuj wzorzec
            stuck_threshold=50,  # Więcej czasu na zrozumienie wzorca
        ),
        "REACH": StrategyParams(
            explore_weight=1.3,
            exploit_weight=2.0,
            level_progress_weight=3.0,
            stuck_threshold=40,
        ),
        "REDUCE": StrategyParams(
            explore_weight=1.0,
            exploit_weight=2.8,  # Priorytet: skoki
            noop_penalty=-2.0,  # Kara za brak ruchu
            stuck_threshold=30,
        ),
        "PLACE": StrategyParams(
            explore_weight=1.2,
            exploit_weight=2.3,
            affordance_weight=2.2,  # Klikaj na skrzynki
            stuck_threshold=45,
        ),
        "FLOW": StrategyParams(
            explore_weight=1.0,
            exploit_weight=2.0,
            pattern_weight=3.0,  # Priorytet: wzorzec przepływu
            stuck_threshold=30,  # Mniej czasu (ograniczone próby)
        ),
        "ALIGN": StrategyParams(
            explore_weight=1.4,
            exploit_weight=1.9,
            affordance_weight=2.4,  # Klikaj na markery
            stuck_threshold=40,
        ),
        "LOCAL_CONSTRAINTS": StrategyParams(
            explore_weight=1.1,
            exploit_weight=2.1,
            affordance_weight=2.6,  # Klikaj na kafelki-klucze
            noop_penalty=-1.5,
            stuck_threshold=35,
        ),
        "UNKNOWN": StrategyParams(
            explore_weight=1.0,
            exploit_weight=2.0,
            stuck_threshold=40,
        ),
    }
    
    return strategies.get(goal_class, strategies["UNKNOWN"])


def get_strategy_hint(goal_class: str) -> str:
    """Zwróć hint taktyczny dla danej klasy gry (do wstrzyknięcia w prompt)."""
    return STRATEGY_HINTS.get(goal_class, STRATEGY_HINTS["UNKNOWN"])


# ---------------------------------------------------------------------------
# Example usage
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    # Test
    grid = [
        [0, 1, 1, 0, 0],
        [1, 2, 2, 1, 0],
        [1, 2, 2, 1, 0],
        [0, 1, 1, 0, 0],
    ]
    
    features = extract_game_features(grid)
    goal_class = detect_goal_class(grid)
    strategy = get_strategy_for_goal_class(goal_class)
    
    print(f"Grid: {grid}")
    print(f"Features: {features}")
    print(f"Detected goal class: {goal_class}")
    print(f"Strategy: {strategy}")


In [ ]:
# LLM policy v7.0
"""
ARC-AGI-3 LLM Agent v7.0 - najlepsza wersja

Łączy:
- Najlepsze elementy z v6.6 (FIELD_GUIDE, LEVEL_NOTES, etc.)
- Counter-picking per game (goal_classifier.py)
- Ewolucja parametrów heurystyki (evolve_parameters.py)
- Ostrzeżenie o ML (inspiracja c87f7a48 v51 league: +1.2pp pool, -1.4pp hard6)
- Deterministyczny ValueGate (heurystyczny, nie ML-based)

Oczekiwany wynik: ~42-50 pkt (vs v6.7 ~40-47)
"""

import json
import os
import re
from dataclasses import dataclass, field
from typing import Any, Optional

from goal_classifier import detect_goal_class, get_strategy_for_goal_class


# ---------------------------------------------------------------------------
# FIELD GUIDE - klasy celów (z v6.6)
# ---------------------------------------------------------------------------

FIELD_GUIDE = """
FIELD GUIDE - Classes of Goals (from GOAL_CATALOG.md):

1. COVER: Cover all target markers with objects
   - Look for: dots, dashed outlines, small markers in moving object's color
   - KEY: "All" means ALL targets covered

2. REACH: Reach all target positions
   - Look for: outlined positions, goal markers, dotted diamonds
   - KEY: All avatars on exits

3. COPY: Make canvas match a pattern
   - Look for: thumbnail in corner, canvas in middle
   - KEY: Large non-interactive area = pattern to copy

4. REDUCE: Leave exactly 1 piece (jump over others like checkers)
   - KEY: Jump over pieces, don't just move them

5. PLACE: Put all objects in target zone
   - KEY: Objects must be in zone AND released (not held)

6. FLOW: Fill all containers without spilling
   - KEY: Set configuration first, then confirm (action 5)

7. ALIGN: Match target positions by color
   - KEY: Alignment by color, not position

8. LOCAL_CONSTRAINTS: Each 3×3 tile-key says which neighbors must have same/different color
   - KEY: Tile-key tells constraint, click changes color

KEY PATTERNS:
- Decreasing counter = STEP LIMIT (zeroing = LOSS, not win!)
- One game = one win condition class (all levels same type)
- "All" means ALL: all targets covered, all avatars on exits, all containers full
- Large non-interactive area = pattern to copy (not something to click)
- Dashed outlines, small markers in moving object's color = targets

CONFIRMATION (CRITICAL):
- In sb26, sp80: CONFIRM (action 5) only ONCE after setting configuration
- Don't click CONFIRM "to test" - limited attempts, errors cost lives
"""


# ---------------------------------------------------------------------------
# Deterministyczny ValueGate (heurystyczny, nie ML-based)
# ---------------------------------------------------------------------------

@dataclass
class HeuristicValueGate:
    """Heurystyczny ValueGate - ocena stanu bez ML.
    
    Inspiracja: c87f7a48 v51 league (acc 0.777 vs base 0.513)
    Ale: pool +1.2pp, hard6 -1.4pp = ML działa na puli, nie na trudnych grach
    
    Wartość:
    - >0.7: dobry stan (kontynuuj)
    - 0.3-0.7: niepewny (eksploruj ostrożnie)
    - <0.3: zły stan (zmień strategię)
    """
    
    def evaluate(self, frame: dict[str, Any], state: Any) -> float:
        """Oceń stan gry heurystycznie."""
        score = 0.5  # Bazowy
        
        # Postęp (im więcej poziomów, tym lepiej)
        levels_completed = frame.get("levels_completed", 0)
        if levels_completed > 0:
            score += 0.1 * levels_completed
        
        # No-op ratio (im mniej, tym lepiej)
        if state.action_count > 0:
            noop_ratio = state.no_op_count / state.action_count
            score -= 0.2 * noop_ratio
        
        # Lives (im więcej, tym lepiej)
        lives = frame.get("lives", 3)
        if lives < 2:
            score -= 0.2
        elif lives > 3:
            score += 0.1
        
        # Actions since level (im mniej, tym lepiej - szybko rozwiązywane poziomy)
        if state.actions_since_level < 20:
            score += 0.1
        elif state.actions_since_level > 50:
            score -= 0.1
        
        # Stuck count (im mniej, tym lepiej)
        if state.stuck_count > 2:
            score -= 0.2
        
        return max(0.0, min(1.0, score))


# ---------------------------------------------------------------------------
# GameState (z v6.6)
# ---------------------------------------------------------------------------

@dataclass
class GameState:
    action_count: int = 0
    actions_since_level: int = 0
    last_grid_hash: str = ""
    stuck_count: int = 0
    strategy: str = "explore"
    notes: list[str] = field(default_factory=list)
    current_level: int = 1
    goal_class: Optional[str] = None
    game_overs: list[str] = field(default_factory=list)
    no_op_count: int = 0
    level_notes: dict[int, str] = field(default_factory=dict)
    value_gate: HeuristicValueGate = field(default_factory=HeuristicValueGate)
    ml_value: float = 0.5


# ---------------------------------------------------------------------------
# Drabinka eskalacji (z v6.6)
# ---------------------------------------------------------------------------

def build_escalation_prompt(stuck_count: int, level: int, actions_since_level: int, goal_class: str = None) -> str:
    """Zbuduj prompt eskalacji w zależności od stuck_count."""
    
    if stuck_count == 1:
        return f"""STRATEGY AUDIT: You've spent {actions_since_level} actions on level {level} without solving it.

Before spending more actions, step back and write a short audit (at most 6 short lines):

1. Which hypotheses about the goal and mechanics have you TESTED?
   - Which were CONFIRMED (✓)?
   - Which were FALSIFIED (✗)?

2. Which board elements, colors, or actions have you NOT yet interacted with?

3. Is the goal DIFFERENT from what you assumed?
   - Re-examine the win condition class (COVER/REACH/COPY/etc.)
   {f"- Current hypothesis: {goal_class}" if goal_class else ""}

4. What is the SINGLE CHEAPEST experiment that best distinguishes the remaining hypotheses?

Write the audit in your reasoning field, then execute the experiment.
"""
    
    elif stuck_count >= 2:
        return f"""STRATEGY RESET: You've spent {actions_since_level} actions on level {level} without solving it, and this is audit #{stuck_count}.

Your current approach is NOT WORKING. ABANDON IT.

1. Look at FIELD GUIDE above. Which goal class fits this game? (COVER/REACH/COPY/REDUCE/PLACE/FLOW/ALIGN)
   {f"- You tried: {goal_class} - CHOOSE DIFFERENT" if goal_class else ""}

2. Choose a DIFFERENT goal class than what you tried before.

3. Start fresh with that goal class.

4. DO NOT repeat the same action patterns you tried before.

Write in your reasoning: "RESETTING STRATEGY: trying [NEW_GOAL_CLASS] instead of [OLD_APPROACH]"
Then execute a completely different approach.
"""
    
    return ""


# ---------------------------------------------------------------------------
# System Prompt
# ---------------------------------------------------------------------------

def build_system_prompt(
    level_notes: Optional[dict[int, str]] = None,
    goal_class: Optional[str] = None,
) -> str:
    """Zbuduj system prompt z FIELD_GUIDE + LEVEL_NOTES + goal class."""
    
    notes_injection = ""
    if level_notes:
        lines = ["\nLEVEL NOTES (from previous levels):"]
        for level in sorted(level_notes.keys()):
            lines.append(f"Level {level}: {level_notes[level]}")
        notes_injection = "\n".join(lines)
    
    goal_injection = ""
    if goal_class:
        goal_injection = f"\n\nDETECTED GOAL CLASS: {goal_class}\n(Strategy adjusted for this goal class)"
    
    return f"""ARC-AGI-3 Puzzle Solver v7.0

{FIELD_GUIDE}

{notes_injection}
{goal_injection}

STRATEGY:
1. ACT FAST - don't think too much (max 60 words reasoning)
2. Track what changes grid = good; what doesn't = bad (no-op detection)
3. If stuck 40 actions without completing level → ESCALATION LADDER
   - Audit #1: audit hypotheses
   - Audit #2+: STRATEGY RESET (choose different goal class from FIELD GUIDE)
4. New object? CLICK it NOW to test (affordance detection)
5. Take NOTES: add "notes" field to actions to remember discoveries
6. After solving a level → write LEVEL X NOTES: paragraph

GOAL PERSISTENCE:
- One game = one win condition class (COVER/REACH/COPY/REDUCE/PLACE/FLOW/ALIGN)
- After solving level 1, remember the goal class
- All levels in same game have same goal type

FAILURE RECOVERY:
- If you get GAME OVER → don't repeat the same mistake
- Write in notes: "GAME OVER at action X, cause: Y"

ANTI-DISSERTATION:
- Keep reasoning SHORT (max 60 words)
- Use "notes" field to remember discoveries across turns
- Batch up to 8 actions at once when confident (if value gate > 0.7)
- Batch up to 2 actions when uncertain (if value gate < 0.3)

OUTPUT FORMAT:
```json
[
  {{"action": "RIGHT", "reasoning": "explore right", "notes": "saw blue gem"}},
  {{"action": "DOWN", "reasoning": "continue down"}}
]
```

For MOUSE: {{"action": "MOUSE", "row": 5, "col": 10, "reasoning": "click target"}}

Keep reasoning SHORT (max 60 words). Use "notes" to remember discoveries.
After solving a level, write "LEVEL X NOTES:" paragraph in your next response.
"""


# ---------------------------------------------------------------------------
# Policy v7.0
# ---------------------------------------------------------------------------

class LLMPolicyV7_0:
    """LLM Agent v7.0 z counter-picking + deterministyczny ValueGate."""
    
    def __init__(self, llm_client: Any):
        self.client = llm_client
        self.state = GameState()
        self.messages = []
        
    def reset(self):
        """Reset stanu na nową grę."""
        self.state = GameState()
        self.messages = [{"role": "system", "content": build_system_prompt()}]
    
    def get_action(self, frame: dict[str, Any]) -> dict[str, Any]:
        """Zwróć akcję dla danego frame'a z counter-picking + ValueGate."""
        self.state.action_count += 1
        self.state.actions_since_level += 1
        
        # Counter-picking: detekcja klasy gry (raz na grę)
        if self.state.goal_class is None and self.state.action_count == 1:
            grid = frame.get("grid", [])
            detected_class = detect_goal_class(grid)
            self.state.goal_class = detected_class
            # Aktualizuj system prompt z detected goal class
            self.messages[0]["content"] = build_system_prompt(
                level_notes=self.state.level_notes,
                goal_class=detected_class,
            )
        
        # Heurystyczny ValueGate
        ml_value = self.state.value_gate.evaluate(frame, self.state)
        self.state.ml_value = ml_value
        
        # Sprawdź czy poziom został ukończony
        if frame.get("levels_completed", 0) > self.state.current_level - 1:
            self.state.current_level = frame["levels_completed"] + 1
            self.state.actions_since_level = 0
            self.state.stuck_count = 0
            # Poproś o LEVEL NOTES
            self.messages.append({
                "role": "system",
                "content": self._build_level_notes_request(frame["levels_completed"])
            })
        
        # Drabinka eskalacji
        if self.state.actions_since_level >= 40:
            self.state.stuck_count += 1
            if self.state.stuck_count % 15 == 0 or self.state.stuck_count <= 2:
                escalation_prompt = build_escalation_prompt(
                    self.state.stuck_count,
                    self.state.current_level,
                    self.state.actions_since_level,
                    self.state.goal_class,
                )
                self.messages.append({
                    "role": "system",
                    "content": escalation_prompt
                })
        
        # No-op detection
        current_grid_hash = str(frame.get("grid", ""))
        if current_grid_hash == self.state.last_grid_hash:
            self.state.no_op_count += 1
        else:
            self.state.no_op_count = max(0, self.state.no_op_count - 1)
        self.state.last_grid_hash = current_grid_hash
        
        # Dodaj frame do messages z ValueGate feedback
        user_msg = self._build_user_message(frame, ml_value)
        self.messages.append({"role": "user", "content": user_msg})
        
        # Deterministyczna temperatura (inspiracja v6.7)
        temperature = 0.6  # Default
        if self.state.stuck_count >= 2:
            temperature = 0.9  # High exploration po STRATEGY RESET
        elif ml_value < 0.3:
            temperature = 0.8  # High exploration gdy zły stan
        elif ml_value > 0.7:
            temperature = 0.4  # Low exploration gdy dobry stan
        
        # Confidence-based max_tokens
        max_tokens = 2000
        if ml_value > 0.8:
            max_tokens = 3000
        elif ml_value < 0.3:
            max_tokens = 1000
        
        # Wywołaj LLM
        response = self.client.chat.completions.create(
            model=os.getenv("LLM_MODEL", "qwen2.5-7b"),
            messages=self.messages,
            temperature=temperature,
            max_tokens=max_tokens,
        )
        
        assistant_msg = response.choices[0].message
        self.messages.append(assistant_msg)
        
        # Wydobądź akcje
        actions = self._parse_actions(assistant_msg.content)
        
        # Zapisz notatki
        if actions and actions[0].get("notes"):
            self.state.notes.append(actions[0]["notes"])
            if len(self.state.notes) > 4:
                self.state.notes = self.state.notes[-4:]
        
        # Zaktualizuj level_notes z odpowiedzi modelu (odporna ekstrakcja z v6.7)
        new_notes = self._extract_level_notes(self.messages[-2:])
        if new_notes:
            self.state.level_notes.update(new_notes)
            self.messages[0]["content"] = build_system_prompt(
                self.state.level_notes,
                self.state.goal_class,
            )
        
        return actions[0] if actions else {"action": "RESET", "reasoning": "fallback"}
    
    def _build_level_notes_request(self, completed_level: int) -> str:
        """Po rozwiązaniu poziomu, poproś model o krótki paragraf NOTATEK."""
        return f"""Level {completed_level} is solved. In your next reply, write one short paragraph 
that starts with `LEVEL {completed_level} NOTES:` and states:
1. The mechanics you verified (what actions do what)
2. The win condition (how to win this level)
3. The action pattern that solved it (sequence of actions)
(At most 4 short sentences, max 200 characters)

The harness keeps these notes for the rest of the game.
"""
    
    def _extract_level_notes(self, messages: list[dict[str, Any]]) -> dict[int, str]:
        """Odporna ekstrakcja LEVEL NOTES (z v6.7).
        
        Skanuje: reasoning, reasoning_content, content
        Case-insensitive (model pisze "Level" nie "LEVEL")
        Filtr długości (min 20 znaków)
        """
        pattern = re.compile(
            r"LEVEL\s*(\d+)\s*NOTES?:\s*(.+?)(?=\n\s*\n|LEVEL\s*\d+\s*NOTES?:|\Z)",
            re.S | re.I,  # re.I = case-insensitive
        )
        found = {}
        limit = 600
        for message in messages:
            if str(message.get("role", "")).strip() != "assistant":
                continue
            # Skanuj reasoning, reasoning_content, content (odporna ekstrakcja z v6.7)
            for key in ("reasoning", "reasoning_content", "content"):
                value = message.get(key)
                if isinstance(value, str):
                    text = value
                elif isinstance(value, list):
                    text = " ".join(
                        part.get("text", "")
                        for part in value
                        if isinstance(part, dict) and isinstance(part.get("text"), str)
                    )
                else:
                    continue
                if "NOTE" not in text.upper():
                    continue
                for match in pattern.finditer(text):
                    note = " ".join(match.group(2).replace("`", " ").split())
                    if len(note) < 20:  # Filtr długości
                        continue
                    try:
                        level = int(match.group(1))
                    except ValueError:
                        continue
                    if 0 < level < 100:
                        found[level] = note[:limit]
        return found
    
    def _build_user_message(self, frame: dict[str, Any], ml_value: float = 0.5) -> str:
        """Zbuduj wiadomość użytkownika z frame'em i ValueGate."""
        grid = frame.get("grid", [])
        available_actions = frame.get("available_actions", [])
        
        grid_str = "\n".join("".join(str(cell) for cell in row) for row in grid)
        
        # ValueGate feedback (deterministyczny, nie ML-based)
        ml_warning = ""
        if ml_value < 0.3:
            ml_warning = f"\n\n⚠️ WARNING: State evaluation low ({ml_value:.2f}). Consider changing strategy."
        elif ml_value > 0.7:
            ml_warning = f"\n\n✓ Good state ({ml_value:.2f}). Continue current approach."
        
        # No-op warning
        no_op_warning = ""
        if self.state.no_op_count >= 3:
            no_op_warning = f"\n\nWARNING: Last {self.state.no_op_count} actions didn't change grid. Try different action."
        
        # Game over history
        game_over_history = ""
        if self.state.game_overs:
            game_over_history = f"\n\nGAME OVER HISTORY: {'; '.join(self.state.game_overs[-3:])}"
        
        # Confidence-based batching hint
        batch_hint = ""
        if ml_value > 0.8:
            batch_hint = "\n\n[HIGH CONFIDENCE: batch up to 8 actions]"
        elif ml_value < 0.3:
            batch_hint = "\n\n[LOW CONFIDENCE: batch only 1-2 actions, explore carefully]"
        
        return f"""Level: {self.state.current_level}
Action: {self.state.action_count}
Actions since level complete: {self.state.actions_since_level}
Available actions: {', '.join(available_actions)}

Grid:
{grid_str}

Recent notes: {'; '.join(self.state.notes[-4:]) if self.state.notes else 'none'}
{ml_warning}{no_op_warning}{game_over_history}{batch_hint}

What action do you take?"""
    
    def _parse_actions(self, content: str) -> list[dict[str, Any]]:
        """Parser z raw_decode (z v6.7 - nie greedy)."""
        # Szukaj JSON w content
        json_match = re.search(r'(\[.*?\]|\{.*?\})', content, re.DOTALL)
        if not json_match:
            return []
        
        try:
            # Użyj raw_decode (nie greedy) jak w v6.7
            decoder = json.JSONDecoder()
            actions, _ = decoder.raw_decode(json_match.group(1))
            if isinstance(actions, list):
                return actions
            elif isinstance(actions, dict):
                return [actions]
        except json.JSONDecodeError:
            pass
        
        return []
    
    def record_game_over(self, cause: str):
        """Zapisz game over dla failure recovery."""
        self.state.game_overs.append(f"Action {self.state.action_count}: {cause}")
        if len(self.state.game_overs) > 5:
            self.state.game_overs = self.state.game_overs[-5:]


if __name__ == "__main__":
    # Test
    class MockLLMClient:
        class Chat:
            class Completions:
                def create(self, **kwargs):
                    class Response:
                        class Choice:
                            class Message:
                                content = '[{"action": "RIGHT", "reasoning": "explore", "notes": "test"}]'
                            message = Message()
                        choices = [Choice()]
                    return Response()
            completions = Completions()
        chat = Chat()
    
    policy = LLMPolicyV7_0(MockLLMClient())
    policy.reset()
    
    frame = {
        "grid": [[0, 1, 2], [3, 4, 5], [6, 7, 8]],
        "available_actions": ["RIGHT", "LEFT", "UP", "DOWN"],
        "levels_completed": 0,
        "current_level": 1,
        "action_count": 0,
        "actions_since_level": 0,
        "lives": 3,
        "score": 0,
        "step_limit": 400,
        "no_op_count": 0,
    }
    
    action = policy.get_action(frame)
    print(f"Action: {action}")
    print(f"Detected goal class: {policy.state.goal_class}")
    print(f"ValueGate: {policy.state.ml_value:.3f}")


In [ ]:
# LLM agent adapter for v7.0
"""ARC-AGI-3 Agent adapter for LLMPolicyV7_0 (counter-picking + ValueGate).

v7.0 improvements over v2:
- Goal classifier (COVER/COPY/REACH/REDUCE/PLACE/FLOW/ALIGN/LOCAL_CONSTRAINTS)
- Counter-picking: strategy hints injected per goal class
- Heuristic ValueGate (no ML weights)
- Escalation ladder (audit #2+ = STRATEGY RESET)
- Robust LEVEL NOTES extraction (case-insensitive, reasoning/reasoning_content)

Plugs LLMPolicyV7_0 into the ARC-AGI-3-Agents reference framework.
"""
from __future__ import annotations

import os
import sys
import time
from threading import RLock
from typing import Any

# Ensure imports work both in competition and locally
_dir = os.path.dirname(os.path.abspath(__file__))
_parent = os.path.dirname(_dir)
for p in (_dir, _parent):
    if p not in sys.path:
        sys.path.insert(0, p)

from arcengine import ActionInput, FrameData, GameAction, GameState
from agents.agent import Agent

try:
    from .llm_policy_v7_0 import LLMPolicyV7_0
    from .grid_utils import format_grid_ascii
    from .goal_classifier import detect_goal_class, get_strategy_hint
except ImportError:
    from llm_policy_v7_0 import LLMPolicyV7_0
    from grid_utils import format_grid_ascii
    from goal_classifier import detect_goal_class, get_strategy_hint


# Minimal FrameInfo for compatibility with v7.0 policy
class FrameInfo:
    def __init__(self, **kwargs):
        for k, v in kwargs.items():
            setattr(self, k, v)


def _extract_grid(frame_data: FrameData) -> tuple[tuple[int, ...], ...]:
    """Extract the numeric grid from a FrameData object."""
    # Try multiple paths to get the grid data
    for attr_path in [
        lambda f: f.raw_state.data,
        lambda f: f.raw_state.planes[0] if f.raw_state.planes else None,
        lambda f: f.planes[0] if hasattr(f, 'planes') and f.planes else None,
    ]:
        try:
            data = attr_path(frame_data)
            if data is None:
                continue
            if hasattr(data, 'tolist'):
                rows = data.tolist()
            else:
                rows = list(data)
            if rows and rows[0]:
                return tuple(tuple(int(c) for c in row) for row in rows)
        except Exception:
            continue
    # Absolute fallback
    return ((0,) * 64,) * 64


def _frame_to_info(frame_data: FrameData) -> FrameInfo:
    """Convert FrameData to our FrameInfo."""
    grid = _extract_grid(frame_data)
    
    # State name
    try:
        state = frame_data.state
        state_name = state.name if hasattr(state, 'name') else str(state)
    except Exception:
        state_name = "PLAYING"
    
    # Levels completed
    try:
        levels_completed = int(frame_data.levels_completed)
    except Exception:
        levels_completed = 0
    
    # Available actions
    available = []
    try:
        for act_id in frame_data.available_actions:
            try:
                act = GameAction.from_id(int(act_id))
                if act.name != "RESET":
                    available.append(act.name)
            except Exception:
                pass
    except Exception:
        available = ["ACTION1", "ACTION2", "ACTION3", "ACTION4", "ACTION5", "ACTION6", "ACTION7"]
    
    # Step number
    try:
        step = int(frame_data.step)
    except Exception:
        step = 0
    
    return FrameInfo(
        grid=grid,
        step=step,
        level=levels_completed + 1,
        state_name=state_name,
        levels_completed=levels_completed,
        available_actions=available,
    )


class LLMAgent(Agent):
    """LLM-powered agent for ARC-AGI-3."""
    
    _ACTION_LOCK = RLock()
    
    ACTION_BUDGET = 400
    MAX_ACTIONS = ACTION_BUDGET
    
    # Soft time limit per game (seconds)
    # 110 games × 4 min = 440 min ≈ 7.3 hours — fits in 9h with margin
    GAME_TIME_LIMIT = 240
    
    def __init__(self, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        self.config = LLMConfig.from_env()
        # Create OpenAI client from config (v7.0 policy uses OpenAI client)
        try:
            from openai import OpenAI
            client = OpenAI(
                base_url=self.config.base_url,
                api_key=self.config.api_key,
                timeout=self.config.timeout,
            )
            # Wrap client to use model_id from config
            class ClientWithModel:
                def __init__(self, client, model_id, temperature, max_output_tokens):
                    self._client = client
                    self._model_id = model_id
                    self._temperature = temperature
                    self._max_output_tokens = max_output_tokens
                class ChatCompletions:
                    def __init__(self, parent):
                        self._parent = parent
                    def create(self, messages, temperature=None, max_tokens=None, **kwargs):
                        return self._parent._client.chat.completions.create(
                            model=self._parent._model_id,
                            messages=messages,
                            temperature=temperature or self._parent._temperature,
                            max_tokens=max_tokens or self._parent._max_output_tokens,
                            **kwargs,
                        )
                class Chat:
                    def __init__(self, parent):
                        self.completions = ClientWithModel.ChatCompletions(parent)
                @property
                def chat(self):
                    return ClientWithModel.Chat(self)
            self.client = ClientWithModel(client, self.config.model_id, self.config.temperature, self.config.max_output_tokens)
        except Exception as e:
            print(f"[LLMAgent] Warning: Failed to create OpenAI client: {e}")
            # Fallback: policy uses MockLLMClient
            self.client = None
        self.policy = LLMPolicyV7_0(self.client)
        self._holds_action_lock = False
        self._game_start_time = time.time()
        self._actions_in_game = 0
        self._first_action = True
    
    @property
    def name(self) -> str:
        return f"{super().name}.llm-v7"
    
    def is_done(self, frames: list[FrameData], latest_frame: FrameData) -> bool:
        """Stop conditions: win, budget, or time limit."""
        if self._actions_in_game >= self.ACTION_BUDGET:
            return True
        try:
            if latest_frame.state is GameState.WIN:
                return True
        except Exception:
            pass
        if time.time() - self._game_start_time > self.GAME_TIME_LIMIT:
            return True
        return False
    
    @staticmethod
    def _to_game_action(proposal: dict) -> GameAction:
        """Convert policy output to GameAction enum."""
        action_name = proposal.get("name", "RESET")
        
        try:
            action = GameAction[action_name]
        except KeyError:
            action = GameAction.RESET
        
        if action.is_complex():
            x = proposal.get("x", 0) or 0
            y = proposal.get("y", 0) or 0
            action.set_data({"x": max(0, min(63, int(x))), "y": max(0, min(63, int(y)))})
        else:
            action.set_data({})
        
        action.reasoning = proposal.get("reasoning", {})
        return action
    
    def choose_action(
        self, frames: list[FrameData], latest_frame: FrameData
    ) -> GameAction:
        """Main entry point called by the framework each turn."""
        self._ACTION_LOCK.acquire()
        self._holds_action_lock = True
        
        try:
            # Build frame dict for v7.0 policy
            grid = _extract_grid(latest_frame)
            
            # Available actions
            available = []
            try:
                for act_id in latest_frame.available_actions:
                    try:
                        act = GameAction.from_id(int(act_id))
                        if act.name != "RESET":
                            available.append(act.name)
                    except Exception:
                        pass
            except Exception:
                available = ["ACTION1", "ACTION2", "ACTION3", "ACTION4", "ACTION5", "ACTION6", "ACTION7"]
            
            # State info
            try:
                state_name = latest_frame.state.name if hasattr(latest_frame.state, "name") else str(latest_frame.state)
            except Exception:
                state_name = "PLAYING"
            
            try:
                levels_completed = int(latest_frame.levels_completed)
            except Exception:
                levels_completed = 0
            
            # Build frame dict that v7.0 policy expects
            frame_dict = {
                "grid": [list(row) for row in grid],
                "available_actions": available,
                "levels_completed": levels_completed,
                "current_level": levels_completed + 1,
                "action_count": self._actions_in_game,
                "actions_since_level": self._actions_in_game,  # simplified
                "lives": 3,  # default
                "score": 0,
                "step_limit": self.ACTION_BUDGET,
                "no_op_count": 0,
                "goal_class": None,
                "game": "unknown",
                "state_name": state_name,
            }
            
            # Initialize on first call
            if self._first_action:
                self._first_action = False
                self._game_start_time = time.time()
                self.policy.reset()
            
            # Auto-reset on GAME_OVER
            if state_name == "GAME_OVER":
                self.policy.record_game_over("Auto-reset: GAME_OVER state")
                action = GameAction.RESET
                action.set_data({})
                action.reasoning = {"auto_reset": True, "game_over": True}
                return action
            
            # Call the LLM policy v7.0
            proposal = self.policy.get_action(frame_dict)
            self._actions_in_game += 1
            
            # Normalize proposal keys (v7.0 returns 'action', we need 'name')
            if "action" in proposal and "name" not in proposal:
                proposal["name"] = proposal["action"]
            
            return self._to_game_action(proposal)
            
        except Exception as e:
            import traceback
            print(f"[LLMAgent] Error: {e}")
            traceback.print_exc()
            action = GameAction.RESET
            action.set_data({})
            action.reasoning = {"error": str(e)}
            return action
        finally:
            if self._holds_action_lock:
                self._holds_action_lock = False
                self._ACTION_LOCK.release()
    
    def take_action(self, action: GameAction):  # type: ignore[override]
        """Execute action and record the result."""
        try:
            frame = super().take_action(action)
            if frame is not None:
                frame.action_input = ActionInput(
                    id=action,
                    data=action.action_data.model_dump(),
                    reasoning=getattr(action, "reasoning", None),
                )
            return frame
        finally:
            if getattr(self, "_holds_action_lock", False):
                self._holds_action_lock = False
                self._ACTION_LOCK.release()
    
    def finalize_diagnostics(self) -> None:
        """Log final stats for this game."""
        elapsed = time.time() - self._game_start_time
        goal_class = getattr(self.policy.state, "goal_class", None)
        stuck = getattr(self.policy.state, "stuck_count", 0)
        levels = getattr(self.policy.state, "current_level", 1) - 1
        ml_value = getattr(self.policy.state, "ml_value", 0.5)
        print(f"[LLMAgent-v7] Game done: "
              f"{self._actions_in_game} actions in {elapsed:.0f}s, "
              f"levels={levels}, goal_class={goal_class}, stuck_count={stuck}, "
              f"ml_value={ml_value:.2f}")


In [ ]:
# Verify setup
from llm_agent_v7 import LLMAgent
print(f'v7.0 agent loaded: {LLMAgent.name}')